# 03 — Context dependence

**The core notebook.** Which perturbations behave differently depending on
immune environment?

Two independent readouts, deliberately not one:

1. **Signature divergence** — correlate each perturbation's log2FC signature in
   IFN-γ and co-culture against its signature in control.
2. **E-distance** (`pertpy`, permutation null) — a model-free answer to "did
   this perturbation do anything at all in this condition."

A perturbation is called context-dependent only if **both** hold: a real
effect somewhere, and divergence between environments. Requiring only
divergence would fill the hit list with noise, because two null signatures are
also uncorrelated. That conjunction is the analytical core of the project.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg = load_config()
panels = load_panels()
P = paths(cfg)
SEED = set_seed(cfg)
apply_style(cfg)

sc.settings.verbosity = 1
print(f"repo: {P.root}")
print(f"seed: {SEED}")


In [ ]:
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna = mdata["rna"]
sig = pd.read_parquet(P.data_processed / "signatures.parquet")
s = cfg["schema"]["obs"]
sig.shape

## 1. Signature divergence

Track two quantities, and do not collapse them: **correlation** (has the effect
been *redirected*?) and **magnitude ratio** (has the same effect been
*amplified*?). These are different phenomena and mixing them muddies the hit
list.

In [ ]:
from src.stats import cross_condition_correlation

ref = cfg["schema"]["conditions"]["reference"]
corr = cross_condition_correlation(sig, reference=ref)
corr.sort_values("pearson_vs_reference").head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for cond, g in corr.groupby("condition"):
    ax.scatter(g["pearson_vs_reference"], np.log2(g["magnitude_ratio"]),
               s=18, alpha=0.7, label=cond,
               color=condition_palette(cfg).get(cond))
ax.axvline(cfg["context"]["call_thresholds"]["max_cross_condition_pearson"],
           ls="--", c="k", lw=1)
ax.axhline(0, ls=":", c="k", lw=1)
ax.set_xlabel(f"Pearson r vs {ref} signature   (low = redirected)")
ax.set_ylabel("log2 magnitude ratio   (high = amplified)")
ax.legend()
savefig(fig, "03_divergence_vs_magnitude", cfg)

## 2. E-distance

Model-free perturbation effect size with a permutation null. This is the
readout that does not inherit the pseudo-replication problem from nb02.

In [ ]:
import pertpy as pt

# PCA space shared across conditions, computed once
rna_n = rna.copy()
sc.pp.normalize_total(rna_n, target_sum=1e4)
sc.pp.log1p(rna_n)
sc.pp.scale(rna_n, max_value=10)
sc.tl.pca(rna_n, n_comps=cfg["context"]["edistance"]["n_pcs"])

# TODO: run pt.tl.Distance(metric="edistance") per condition, with
# distance.test(...) for the permutation p-value. Subsample per
# config -> context.edistance.subsample_cells_per_group; this is the
# slowest step in the project.
print("[stub] E-distance per condition -> edist_df"
      " with columns [perturbation, condition, edist, edist_pvalue]")

## 3. Call hits

In [ ]:
from src.stats import call_context_dependent

# hits = call_context_dependent(corr, edist_df, cfg)
# hits[hits["context_dependent"]].to_csv(P.tables / "03_context_dependent_hits.csv")
# hits[hits["context_dependent"]].head(30)
print("[stub] requires edist_df from section 2")

## 4. Perturbation modules

Cluster perturbations by their context-dependence profile. The check that the
clustering is meaningful: IFN-γ receptor and JAK/STAT components should group
together without being told to.

In [ ]:
# TODO: Leiden on the signature manifold (config -> context.clustering),
# then annotate clusters against panels["gene_sets"].
print("[stub] perturbation modules")

## 5. Pathway enrichment on the hit set

In [ ]:
# TODO: decoupler + MSigDB hallmark/reactome per config -> msigdb
print("[stub] enrichment")

## 6. STRING PPI on hits

Reuse the approach from the Norman project — already written, cheap, and a good visual.

In [ ]:
# TODO: STRING network of context-dependent hits
print("[stub] PPI")